## 1. Imports and numerical settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scqubits as scq
from scipy.linalg import cosm


# Output path
FIGURE_FOLDER = Path("figures")
FIGURE_FOLDER.mkdir(exist_ok=True)

# Circuit-energy parameters in GHz.
EJ = 8.9
EC = 2.5
EL = 0.5

# Convergence settings used throughout the analysis.
N_REF = 200
CUTOFF_VALUES = np.arange(10, 101, 2)
TOLERANCE_GHZ = 1e-3  # 1 MHz

## 2. Fluxonium Hamiltonian

In [ ]:
def calculate_fluxonium_energies(
    EJ,
    EC,
    EL,
    cutoff,
    flux_fraction,
    truncated_dim,
):
    """Return the lowest fluxonium eigenenergies in a truncated oscillator basis."""
    # Convert reduced external flux Phi_ext/Phi_0 to phase.
    phi_ext = 2 * np.pi * flux_fraction
    # Harmonic LC spacing and zero-point phase fluctuation.
    f_lc = np.sqrt(8 * EC * EL)
    phi_zpf = (2 * EC / EL) ** 0.25

    oscillator_levels = np.arange(cutoff)
    H0 = np.diag(f_lc * (oscillator_levels + 0.5))

    # Construct phi = phi_zpf(a + a^dagger) in the truncated basis.
    off_diagonal = phi_zpf * np.sqrt(np.arange(1, cutoff))
    phi_op = (
        np.diag(off_diagonal, k=1)
        + np.diag(off_diagonal, k=-1)
    )

    # Evaluate the Josephson cosine as a matrix function.
    junction_phase_op = phi_op - phi_ext * np.eye(cutoff)
    H = H0 - EJ * cosm(junction_phase_op)

    energies = np.linalg.eigvalsh(H)
    return energies[:truncated_dim]

## 3. Convergence utilities

In [ ]:
def calculate_relative_energy(
    EJ,
    EC,
    EL,
    cutoff,
    flux_fraction,
    state,
):
    """Return E_state - E_0 for one basis cutoff and external flux."""
    energies = calculate_fluxonium_energies(
        EJ,
        EC,
        EL,
        cutoff,
        flux_fraction,
        truncated_dim=state + 1,
    )
    return energies[state] - energies[0]


def calculate_state_convergence_errors(
    EJ,
    EC,
    EL,
    flux_fraction,
    cutoff_values,
    N_ref,
    state,
):
    """Compare a relative energy with the value obtained at the reference cutoff."""
    # Use the large-cutoff result as the numerical reference.
    reference_energy = calculate_relative_energy(
        EJ,
        EC,
        EL,
        N_ref,
        flux_fraction,
        state,
    )

    errors = []
    for cutoff in cutoff_values:
        energy = calculate_relative_energy(
            EJ,
            EC,
            EL,
            cutoff,
            flux_fraction,
            state,
        )
        errors.append(abs(energy - reference_energy))

    return np.array(errors)


def calculate_state_minimum_cutoff(
    EJ,
    EC,
    EL,
    flux_fraction,
    cutoff_values,
    N_ref,
    tolerance,
    state,
):
    """Return the first cutoff after which all tested errors remain below tolerance."""
    errors = calculate_state_convergence_errors(
        EJ,
        EC,
        EL,
        flux_fraction,
        cutoff_values,
        N_ref,
        state,
    )

    # Require persistent convergence: every larger tested cutoff must
    # remain below the tolerance, not just the first crossing.
    for i, cutoff in enumerate(cutoff_values):
        if np.all(errors[i:] < tolerance):
            return cutoff

    return None

## 4. Figure 3 — convergence versus basis cutoff

In [ ]:
# Representative flux biases used to compare cutoff convergence.
representative_fluxes = [0.0, 0.25, 0.5]

plt.figure(figsize=(8, 5))

for flux_fraction in representative_fluxes:
    errors = calculate_state_convergence_errors(
        EJ,
        EC,
        EL,
        flux_fraction,
        CUTOFF_VALUES,
        N_REF,
        state=1,
    )

    plt.semilogy(
        CUTOFF_VALUES,
        errors,
        marker="o",
        markersize=4,
        label=rf"$\Phi_{{\mathrm{{ext}}}}/\Phi_0={flux_fraction}$",
    )

# Show the 1 MHz convergence threshold used in the paper.
plt.axhline(
    TOLERANCE_GHZ,
    linestyle="--",
    label="1 MHz tolerance",
)

plt.xlabel(r"Basis cutoff $N$")
plt.ylabel(r"Absolute error in $f_{01}$ (GHz)")
plt.title(r"Convergence of the $0\rightarrow1$ transition frequency")
plt.legend()
plt.grid()
plt.tight_layout()

plt.savefig(
    FIGURE_FOLDER / "convergence_vs_cutoff.pdf",
    bbox_inches="tight",
)
plt.show()

## 5. Figure 4 — basis-truncation error across external flux

In [ ]:
# Sweep two flux periods to test convergence across external flux.
convergence_flux_values = np.linspace(0, 2, 51)

error_matrix = np.zeros(
    (len(CUTOFF_VALUES), len(convergence_flux_values))
)

for j, flux_fraction in enumerate(convergence_flux_values):
    error_matrix[:, j] = calculate_state_convergence_errors(
        EJ,
        EC,
        EL,
        flux_fraction,
        CUTOFF_VALUES,
        N_REF,
        state=1,
    )

# Apply a numerical floor before taking log10 so zero roundoff values
# do not produce -inf in the heatmap.
safe_error_matrix = np.maximum(error_matrix, 1e-15)
log_error_matrix = np.log10(safe_error_matrix)

plt.figure(figsize=(9, 6))

image = plt.imshow(
    log_error_matrix,
    origin="lower",
    aspect="auto",
    extent=[
        convergence_flux_values[0],
        convergence_flux_values[-1],
        CUTOFF_VALUES[0],
        CUTOFF_VALUES[-1],
    ],
)

cbar = plt.colorbar(image)
cbar.set_label(r"$\log_{10}(\epsilon_N/\mathrm{GHz})$")

plt.xlabel(r"External flux $\Phi_{\mathrm{ext}}/\Phi_0$")
plt.ylabel(r"Basis cutoff $N$")
plt.title(r"Basis-truncation error in the $0\rightarrow1$ transition")
plt.tight_layout()

plt.savefig(
    FIGURE_FOLDER / "convergence_error_heatmap.pdf",
    bbox_inches="tight",
)
plt.show()

## 6. Figure 5 — state-dependent convergence

In [ ]:
# Compare convergence of higher relative eigenenergies at half flux.
flux_fraction = 0.5
states = [1, 2, 3, 4]

plt.figure(figsize=(8, 5))

for state in states:
    errors = calculate_state_convergence_errors(
        EJ,
        EC,
        EL,
        flux_fraction,
        CUTOFF_VALUES,
        N_REF,
        state,
    )

    plt.semilogy(
        CUTOFF_VALUES,
        errors,
        marker="o",
        markersize=4,
        label=rf"$E_{state}-E_0$",
    )

# Show the 1 MHz convergence threshold used in the paper.
plt.axhline(
    TOLERANCE_GHZ,
    linestyle="--",
    label="1 MHz tolerance",
)

plt.xlabel(r"Basis cutoff $N$")
plt.ylabel("Absolute energy error (GHz)")
plt.title(
    r"State-dependent convergence at "
    r"$\Phi_{\mathrm{ext}}/\Phi_0=0.5$"
)
plt.legend()
plt.grid()
plt.tight_layout()

plt.savefig(
    FIGURE_FOLDER / "state_convergence.pdf",
    bbox_inches="tight",
)
plt.show()

## 7. Harmonic-limit validation

In [ ]:
# Setting EJ = 0 removes the Josephson term and gives the exactly
# solvable harmonic-oscillator limit.
cutoff_test = 40
num_levels = 6
flux_test = 0.0
EJ_test = 0.0

energies_num = calculate_fluxonium_energies(
    EJ_test,
    EC,
    EL,
    cutoff_test,
    flux_test,
    num_levels,
)
energies_num_rel = energies_num - energies_num[0]

k = np.arange(num_levels)
energies_exact = k * np.sqrt(8 * EC * EL)
errors = np.abs(energies_num_rel - energies_exact)

print("k    Numerical (GHz)    Analytical (GHz)    Error (GHz)")
for i in range(num_levels):
    print(
        f"{i:<4}"
        f"{energies_num_rel[i]:<19.10f}"
        f"{energies_exact[i]:<20.10f}"
        f"{errors[i]:.3e}"
    )

## 8. Comparison with scqubits

In [ ]:
# Cross-check the custom matrix construction against scqubits using
# identical circuit parameters, flux values, and basis cutoff.
comparison_fluxes = [0.0, 0.25, 0.5]
cutoff_test = 40
num_levels = 5

print("flux    k    custom (GHz)    scqubits (GHz)    abs. error (GHz)")

for flux_fraction in comparison_fluxes:
    custom = calculate_fluxonium_energies(
        EJ,
        EC,
        EL,
        cutoff_test,
        flux_fraction,
        num_levels,
    )
    custom = custom - custom[0]

    fluxonium = scq.Fluxonium(
        EJ=EJ,
        EC=EC,
        EL=EL,
        flux=flux_fraction,
        cutoff=cutoff_test,
    )
    scq_energies = fluxonium.eigenvals(evals_count=num_levels)
    scq_energies = scq_energies - scq_energies[0]

    for k in range(1, num_levels):
        error = abs(custom[k] - scq_energies[k])
        print(
            f"{flux_fraction:4.2f}    "
            f"{k}    "
            f"{custom[k]:.10f}    "
            f"{scq_energies[k]:.10f}    "
            f"{error:.3e}"
        )

## 9. Low-lying energy spectrum

In [ ]:
cutoff = 40
num_levels = 6
spectrum_flux_values = np.linspace(0, 1, 201)

# Subtract E0 at each flux so the spectrum is shown in relative energies.
relative_energies = []

for flux_fraction in spectrum_flux_values:
    energies = calculate_fluxonium_energies(
        EJ,
        EC,
        EL,
        cutoff,
        flux_fraction,
        num_levels,
    )
    relative_energies.append(energies - energies[0])

relative_energies = np.array(relative_energies)

plt.figure(figsize=(8, 5.5))

for k in range(1, num_levels):
    plt.plot(
        spectrum_flux_values,
        relative_energies[:, k],
        label=fr"$E_{k}-E_0$",
    )

plt.xlabel(r"External flux $\Phi_{\mathrm{ext}}/\Phi_0$")
plt.ylabel(r"Relative energy $(E_k-E_0)/h$ (GHz)")
plt.title("Low-Lying Fluxonium Energy Spectrum")
plt.xlim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_FOLDER / "fluxonium_spectrum.pdf",
    bbox_inches="tight",
)
plt.show()

## 10. Figure 6 — required cutoff across flux and state

In [ ]:
states = [1, 2, 3, 4]

# Store the minimum persistently converged cutoff for each flux and state.
minimum_cutoff_matrix = np.full(
    (len(convergence_flux_values), len(states)),
    np.nan,
)

for j, flux_fraction in enumerate(convergence_flux_values):
    print(
        f"Calculating flux {j + 1}/{len(convergence_flux_values)}: "
        f"{flux_fraction:.2f}"
    )

    # Compute the reference spectrum once for each external-flux value.
    reference_energies = calculate_fluxonium_energies(
        EJ,
        EC,
        EL,
        N_REF,
        flux_fraction,
        truncated_dim=max(states) + 1,
    )
    reference_relative = reference_energies - reference_energies[0]

    state_errors = np.zeros((len(CUTOFF_VALUES), len(states)))

    for i, cutoff in enumerate(CUTOFF_VALUES):
        energies = calculate_fluxonium_energies(
            EJ,
            EC,
            EL,
            cutoff,
            flux_fraction,
            truncated_dim=max(states) + 1,
        )
        relative = energies - energies[0]

        for s, state in enumerate(states):
            state_errors[i, s] = abs(
                relative[state] - reference_relative[state]
            )

    # Apply the same persistent-convergence criterion to every state.
    for s in range(len(states)):
        for i, cutoff in enumerate(CUTOFF_VALUES):
            if np.all(state_errors[i:, s] < TOLERANCE_GHZ):
                minimum_cutoff_matrix[j, s] = cutoff
                break

In [ ]:
state_labels = [
    r"$E_1-E_0$",
    r"$E_2-E_0$",
    r"$E_3-E_0$",
    r"$E_4-E_0$",
]

# Summarize how the required cutoff varies across the tested flux range.
medians = np.nanmedian(minimum_cutoff_matrix, axis=0)
minimums = np.nanmin(minimum_cutoff_matrix, axis=0)
maximums = np.nanmax(minimum_cutoff_matrix, axis=0)

yerr = np.vstack([
    medians - minimums,
    maximums - medians,
])

x = np.arange(len(states))

plt.figure(figsize=(7, 4.5))
plt.errorbar(
    x,
    medians,
    yerr=yerr,
    fmt="o",
    capsize=6,
    markersize=7,
)

plt.xticks(x, state_labels)
plt.ylabel(r"Required basis cutoff $N_{\min}$")
plt.title(r"Basis cutoff required for $1$ MHz accuracy")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_FOLDER / "minimum_cutoff_summary.pdf",
    bbox_inches="tight",
)
plt.show()

## 11. Compact convergence summary

In [ ]:
print("State    Maximum required N    Flux where maximum first occurs")
print("--------------------------------------------------------------")

# Report the worst-case required cutoff for each tested relative energy.
state_maximum_cutoffs = []

for s, state in enumerate(states):
    values = minimum_cutoff_matrix[:, s]

    if np.all(np.isnan(values)):
        print(f"E{state}-E0    Not converged within tested range")
        continue

    maximum_N = int(np.nanmax(values))
    first_index = np.where(values == maximum_N)[0][0]
    first_flux = convergence_flux_values[first_index]

    state_maximum_cutoffs.append(maximum_N)

    print(
        f"E{state}-E0"
        f"        {maximum_N:<20}"
        f"{first_flux:.2f}"
    )

if state_maximum_cutoffs:
    overall_cutoff = max(state_maximum_cutoffs)
    print(
        "\nConservative cutoff for all tested states and flux values:"
    )
    print(f"N = {overall_cutoff}")